# QUELLE RÉPARTITION DES PRÊTS À LA MÉDIATHÈQUE ?

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime as dt
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import kiblib.utils.lucas_sns_params
from kiblib.utils.db import DbConn

In [2]:
maj = dt.today().strftime('%d-%m-%Y à %Hh%M' )
print(f"Dernière mise à jour le {maj}")

Dernière mise à jour le 18-11-2026 à 11h01


In [3]:
db_conn = DbConn().create_engine()

In [4]:
query = """SELECT *
FROM koha_prod.items i
LEFT JOIN koha_prod.authorised_values av ON av.authorised_value = i.location"""

In [5]:
items = pd.read_sql(query,db_conn)

In [6]:
cols2print = ["notforloan","damaged","damaged_on","barcode","location","ccode"]

In [7]:
# EXEMPLAIRES
items["Exemplaires"] = 1

In [8]:
# ACQUISITIONS
items.loc[items["dateaccessioned"].astype("Datetime64").between("2026-01-01","2026-12-31"),"Acquisitions"] = 1

In [9]:
# DOCUMENTS A DISPOSITION DU PUBLIC
pret_direct = ["MED0C","MED1A","MED2A","MED3A"]
pret_indirect = ["MED0A","MED2C","MED2D","MED3C","MED3D","MED3E","MED3F","MED3G","MED3H","MED3I"]
consultation = ["MED0B","MED2B"]

# Empruntables
items.loc[items["notforloan"]==0,"Exemplaires empruntables"] = 1

# En accès direct
items.loc[~(items["location"].isin(pret_indirect))&
          ~(items["location"].isin(consultation)),
          "Exemplaires en accès direct"] = 1

# Consultation uniquement
items.loc[items["location"].isin(consultation),
         "Exemplaires en consultation sur place uniquement"] = 1

# En accès indirect
items.loc[items["location"].isin(pret_indirect),
         "Exemplaires en accès indirect"] = 1

In [10]:
# DOCUMENTS DANS LE CIRCUIT INTERNE

items.loc[items["notforloan"]==-1,"Exemplaires en commande"] = 1
items.loc[items["notforloan"]==-2,"Exemplaires en traitement"] = 1
items.loc[items["damaged"]!=0,"Exemplaires abîmés"] = 1
items.loc[items["notforloan"]==-4,"Exemplaires en réparation"] = 1
items.loc[items["notforloan"]==5,"Exemplaires en reliure"] = 1
items.loc[items["itemlost"]==2,"Exemplaires perdus"] = 1
items.loc[items["itemlost"]==1,"Exemplaires non-restitués"] = 1

In [11]:
# DOCUMENTS ÉLIMINÉS

items.loc[(items["notforloan"]==4) &
          (items["itemlost"]==1),"Exemplaires éliminés non-restitués"] = 1

items.loc[(items["notforloan"]==4)&
          (items["itemlost"]==2),
          "Exemplaires éliminés perdus"] = 1


items.loc[(items["notforloan"]==4)&
          (items["withdrawn"]==1),
          "Exemplaires éliminés desherbés"] = 1


items.loc[(items["notforloan"]==4)&
          (items["damaged"]==1),
          "Exemplaires éliminés abîmés"] = 1

In [12]:
site_med = ["MED0B",
            "MED0C",
            "MED1A",
            "MED2A",
            "MED2B",
            "MED2C",
            "MED2D",
            "MED3A",
            "MED3B",
            "MED3C",
            "MED3D",
            "MED3E",
            "MED3F",
            "MED3G",
            "MED3H",
            "MED3I",
            "MED3J",
            "MED3K",
            "MED3L",
            "MED3M",
            "MED3N",
            "MED3O",
            "MED3P",
            "MED3Q",
            "MED3R",
            "MED3S",
            "MED3T"]

site_bus = ["BUS1A"]

site_col = ["MED0A"]

In [13]:
# SITES

# Médiathèque
items.loc[(items["homebranch"]=="MED")&
          (items["location"].isin(site_med)),"Site"] = "Médiatèque"

# Zébron
items.loc[items["homebranch"]=="BUS","Site"] = "Zébron"

# Collectivités
items.loc[items["location"]=="MED0A","Site"] = "Collectivités"

In [14]:
cols2keep = ["Site",
             "ccode",
             "location",
             "lib",
             'Exemplaires', 'Acquisitions',
             'Exemplaires empruntables', 'Exemplaires en accès direct',
             'Exemplaires en consultation sur place uniquement',
             'Exemplaires en accès indirect', 'Exemplaires en commande',
             'Exemplaires en traitement', 'Exemplaires abîmés',
             'Exemplaires en réparation', 'Exemplaires en reliure',
             'Exemplaires perdus', 'Exemplaires non-restitués',
             'Exemplaires éliminés non-restitués', 'Exemplaires éliminés perdus',
             'Exemplaires éliminés desherbés', 'Exemplaires éliminés abîmés']

indicateurs = ['Exemplaires', 'Acquisitions',
              'Exemplaires empruntables', 'Exemplaires en accès direct',
              'Exemplaires en consultation sur place uniquement',
              'Exemplaires en accès indirect', 'Exemplaires en commande',
              'Exemplaires en traitement', 'Exemplaires abîmés',
              'Exemplaires en réparation', 'Exemplaires en reliure',
              'Exemplaires perdus', 'Exemplaires non-restitués',
              'Exemplaires éliminés non-restitués', 'Exemplaires éliminés perdus',
              'Exemplaires éliminés desherbés', 'Exemplaires éliminés abîmés']

In [15]:
analyse = items[cols2keep]

In [16]:
analyse

,Site,ccode,location,lib,Exemplaires,Acquisitions,Exemplaires empruntables,Exemplaires en accès direct,Exemplaires en consultation sur place uniquement,Exemplaires en accès indirect,...,Exemplaires en traitement,Exemplaires abîmés,Exemplaires en réparation,Exemplaires en reliure,Exemplaires perdus,Exemplaires non-restitués,Exemplaires éliminés non-restitués,Exemplaires éliminés perdus,Exemplaires éliminés desherbés,Exemplaires éliminés abîmés
0,Collectivités,JABEFAE,MED0A,Magasin collectivités,1,NaN,1.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Collectivités,JABEFAE,MED0A,Magasin collectivités,1,NaN,1.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN
2,Collectivités,JABEFAE,MED0A,Magasin collectivités,1,NaN,1.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Collectivités,JABEFAE,MED0A,Magasin collectivités,1,NaN,1.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Collectivités,JABEFAE,MED0A,Magasin collectivités,1,NaN,1.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268024,Collectivités,JABPFAF,MED0A,Magasin collectivités,1,1.0,NaN,NaN,NaN,1.0,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
268025,Collectivités,JABPFAF,MED0A,Magasin collectivités,1,1.0,NaN,NaN,NaN,1.0,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
268026,Médiatèque,AAPBDAD,MED0C,Rez-de-chaussée,1,1.0,NaN,1.0,NaN,NaN,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
268027,Collectivités,JABPFAF,MED0A,Magasin collectivités,1,1.0,NaN,NaN,NaN,1.0,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# FILTRE SUR LES EXEMPLAIRES EN PRÊT DIRECT 
# SITE MEDIATHÈQUE
analyse_med = analyse[(analyse["Site"]=="Médiatèque")
                     #&(analyse["location"].isin(pret_direct))
                     ]

In [18]:
analyse_med = analyse_med.pivot_table(index=["location","lib"],values=indicateurs,aggfunc=sum)

In [25]:
analyse_med

,,Acquisitions,Exemplaires,Exemplaires abîmés,Exemplaires empruntables,Exemplaires en accès direct,Exemplaires en accès indirect,Exemplaires en commande,Exemplaires en consultation sur place uniquement,Exemplaires en reliure,Exemplaires en réparation,Exemplaires en traitement,Exemplaires non-restitués,Exemplaires perdus,Exemplaires éliminés abîmés,Exemplaires éliminés desherbés,Exemplaires éliminés non-restitués,Exemplaires éliminés perdus
location,lib,,,,,,,,,,,,,,,,,
MED0B,Espace Périodiques,2.0,5,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
MED0C,Rez-de-chaussée,704.0,11033,18.0,10323.0,11033.0,0.0,17.0,0.0,0.0,4.0,61.0,380.0,14.0,11.0,14.0,336.0,14.0
MED1A,1er étage,4108.0,67285,585.0,63684.0,67285.0,0.0,231.0,0.0,0.0,70.0,174.0,1203.0,134.0,485.0,770.0,1013.0,122.0
MED2A,2e étage - Jeunesse,3379.0,40384,1419.0,34272.0,40384.0,0.0,0.0,0.0,1.0,234.0,349.0,4048.0,65.0,1102.0,190.0,3765.0,48.0
MED2B,2e étage - Salle d'étude,40.0,1478,0.0,10.0,0.0,0.0,0.0,1478.0,0.0,0.0,30.0,3.0,0.0,0.0,0.0,3.0,0.0
MED2C,Magasin jeunesse,22.0,962,3.0,700.0,0.0,962.0,0.0,0.0,0.0,2.0,1.0,23.0,1.0,0.0,0.0,22.0,1.0
MED2D,Magasin disco 2e étage,80.0,15847,0.0,11654.0,0.0,15847.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
MED3A,3e étage,811.0,15392,50.0,14242.0,15392.0,0.0,0.0,0.0,0.0,28.0,29.0,761.0,15.0,17.0,238.0,705.0,11.0
MED3B,Espace Multimédia,3.0,3,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
analyse_med.reindex(cols2keep,axis=1)[["Exemplaires abîmés","Exemplaires en réparation","Exemplaires éliminés abîmés"]]

,,Exemplaires abîmés,Exemplaires en réparation,Exemplaires éliminés abîmés
location,lib,,,
MED0B,Espace Périodiques,0.0,0.0,0.0
MED0C,Rez-de-chaussée,18.0,4.0,11.0
MED1A,1er étage,585.0,70.0,485.0
MED2A,2e étage - Jeunesse,1419.0,234.0,1102.0
MED2B,2e étage - Salle d'étude,0.0,0.0,0.0
MED2C,Magasin jeunesse,3.0,2.0,0.0
MED2D,Magasin disco 2e étage,0.0,0.0,0.0
MED3A,3e étage,50.0,28.0,17.0
MED3B,Espace Multimédia,0.0,0.0,0.0


- Exemplaires
    * Total
    * Empruntables
        * En accès direct
        * En accès indirect
    * Consultation sur place uniquement

- Acquisitions
- Exemplaires dans le circuit du document
    * En commande
    * En traitement
    * En réparation
    * Perdus
    * Non restitués

- Exemplaires sortis des collections
    * Désherbé
    * Non-restitué
    * Perdus
    * Abîmés

# DE QUOI SE COMPOSENT LES COLLECTIONS ?

## Quel est le nombre d'exemplaires par espaces ?

In [31]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Exemplaires",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Exemplaires",ascending=False)

,,Exemplaires
location,lib,
MED1A,1er étage,67285
MED2A,2e étage - Jeunesse,40384
MED3E,Mag Pôle adultes °8,21014
MED2D,Magasin disco 2e étage,15847
MED3A,3e étage,15392
MED3I,Mag Patrimoine °8,14317
MED0C,Rez-de-chaussée,11033
MED3C,Magasin disco 3e étage,10689
MED3T,Mag Patrimoine,7174


## Quel est le nombre d'exemplaire empruntables...
### En accès direct

In [21]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Exemplaires empruntables",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Exemplaires empruntables",ascending=False).loc[analyse_med["Exemplaires empruntables"]!=0]

,,Exemplaires empruntables
location,lib,
MED1A,1er étage,63684
MED2A,2e étage - Jeunesse,34272
MED3E,Mag Pôle adultes °8,20972
MED3A,3e étage,14242
MED2D,Magasin disco 2e étage,11654
MED0C,Rez-de-chaussée,10323
MED3C,Magasin disco 3e étage,3063
MED3D,Mag Bibliothèque professionnelle,2071
MED3H,Mag Pôle adultes BD,1780


### En accès indirect

In [22]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Exemplaires en accès direct",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Exemplaires en accès direct",ascending=False).loc[analyse_med["Exemplaires en accès direct"]!=0]

,,Exemplaires en accès direct
location,lib,
MED1A,1er étage,67285
MED2A,2e étage - Jeunesse,40384
MED3A,3e étage,15392
MED0C,Rez-de-chaussée,11033
MED3T,Mag Patrimoine,7174
MED3J,Mag Patrimoine °4,6239
MED3N,Mag Patrimoine BR 4,3701
MED3M,Mag Patrimoine BR 8,3575
MED3O,Mag Patrimoine RES 8,1998


In [23]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Exemplaires en accès indirect",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Exemplaires en accès indirect",ascending=False).loc[analyse_med["Exemplaires en accès indirect"]!=0]

,,Exemplaires en accès indirect
location,lib,
MED3E,Mag Pôle adultes °8,21014
MED2D,Magasin disco 2e étage,15847
MED3I,Mag Patrimoine °8,14317
MED3C,Magasin disco 3e étage,10689
MED3D,Mag Bibliothèque professionnelle,2123
MED3H,Mag Pôle adultes BD,1798
MED3G,Mag Pôle adultes fol,1010
MED2C,Magasin jeunesse,962
MED3F,Mag Pôle adultes °4,553


## Quel est le nombre d'exemplaires en consultation sur place uniquement ?

In [24]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Exemplaires en consultation sur place uniquement",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Exemplaires en consultation sur place uniquement",ascending=False).loc[analyse_med["Exemplaires en consultation sur place uniquement"]!=0]

,,Exemplaires en consultation sur place uniquement
location,lib,
MED2B,2e étage - Salle d'étude,1478
MED0B,Espace Périodiques,5


## Quel est le nombre d'acquisitions* ?
> (*Acquisitions : nombre de documents créés dans la base pour l'année en cours)

In [33]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Acquisitions",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Acquisitions",ascending=False).loc[analyse_med["Acquisitions"]!=0]

,,Acquisitions
location,lib,
MED1A,1er étage,4108
MED2A,2e étage - Jeunesse,3379
MED3T,Mag Patrimoine,1634
MED3A,3e étage,811
MED0C,Rez-de-chaussée,704
MED2D,Magasin disco 2e étage,80
MED3M,Mag Patrimoine BR 8,52
MED3I,Mag Patrimoine °8,49
MED2B,2e étage - Salle d'étude,40


## Exemplaires dans le circuit du document
### Quel est le nombre d'exemplaire `en commande` ?

In [34]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Exemplaires en commande",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Exemplaires en commande",ascending=False).loc[analyse_med["Exemplaires en commande"]!=0]

,,Exemplaires en commande
location,lib,
MED1A,1er étage,231
MED0C,Rez-de-chaussée,17


### Quel est le nombre d'exemplaires `En traitement` ?

In [35]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Exemplaires en traitement",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Exemplaires en traitement",ascending=False).loc[analyse_med["Exemplaires en traitement"]!=0]

,,Exemplaires en traitement
location,lib,
MED2A,2e étage - Jeunesse,349
MED1A,1er étage,174
MED0C,Rez-de-chaussée,61
MED3M,Mag Patrimoine BR 8,33
MED2B,2e étage - Salle d'étude,30
MED3A,3e étage,29
MED3N,Mag Patrimoine BR 4,16
MED3I,Mag Patrimoine °8,11
MED3P,Mag Patrimoine RES 4,8


### Quel est le nombre d'exemplaires `Perdus` ?

In [46]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Exemplaires perdus",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Exemplaires perdus",ascending=False).loc[analyse_med["Exemplaires perdus"]!=0]

,,Exemplaires perdus
location,lib,
MED1A,1er étage,134
MED2A,2e étage - Jeunesse,65
MED3C,Magasin disco 3e étage,32
MED3A,3e étage,15
MED0C,Rez-de-chaussée,14
MED3E,Mag Pôle adultes °8,12
MED3G,Mag Pôle adultes fol,2
MED3D,Mag Bibliothèque professionnelle,1
MED2C,Magasin jeunesse,1


### Quel est le nombre d'exemplaires `Non-restitués` ?

In [47]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Exemplaires non-restitués",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Exemplaires non-restitués",ascending=False).loc[analyse_med["Exemplaires non-restitués"]!=0]

,,Exemplaires non-restitués
location,lib,
MED2A,2e étage - Jeunesse,4048
MED1A,1er étage,1203
MED3A,3e étage,761
MED0C,Rez-de-chaussée,380
MED3E,Mag Pôle adultes °8,25
MED2C,Magasin jeunesse,23
MED3D,Mag Bibliothèque professionnelle,5
MED2B,2e étage - Salle d'étude,3
MED3H,Mag Pôle adultes BD,2


## Exemplaires sortis des collections (en cours d'année)
### Quel est le nombre d'exemplaires sortis des collections `desherbés` ?

### Quel est le nombre d'exemplaires sortis des collections `non-restitués` ?

### Quel est le nombre d'exemplaires sortis des collections `perdus` ?

### Quel est le nombre d'exemplaires sortis des collections `Abîmés` ?